In [3]:

# ============================================================
# AGAINST ALL ODDS — WEEK 6 FEATURE ENGINEERING
# ============================================================
# Goal:
#   Create three model-ready datasets for NBA, NFL, and Premier League.
#
# Core Week 6 requirements:
#   - Team strength: win percentage, Elo, point/goal differential
#   - Recent form: last 3, 5, and 10 games
#   - Offensive features
#   - Defensive features
#   - Context: home advantage, rest, opponent strength
#   - Matchup differences: ELO_DIFF, FORM_DIFF, OFFENSE_DIFF,
#     DEFENSE_DIFF, REST_DIFF
#   - Use shift(1) before rolling/expanding calculations
#   - Save three leakage-safe model-ready datasets
#
# This cell reloads the Week 3–5 cleaned CSV files from disk instead
# of relying on variables left in memory by earlier notebook cells.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import zipfile
import warnings

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_TITLE = "Against All Odds"
WEEK_NUMBER = 6
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ELO_INITIAL = 1500.0
ELO_K = {
    "NBA": 20.0,
    "NFL": 20.0,
    "Premier League": 20.0,
}

# ------------------------------------------------------------
# 1. Locate the project folder
# ------------------------------------------------------------

def running_in_colab():
    return "google.colab" in sys.modules


def choose_project_root():
    cwd = Path.cwd().resolve()

    candidates = [
        cwd if cwd.name.lower().replace("_", "-") in {
            "againstallodds",
            "against-all-odds",
        } else None,
        Path("/content/againstallodds"),
        Path("/content/Against-All-Odds"),
        Path("/content/drive/MyDrive/againstallodds"),
        Path("/content/drive/MyDrive/Against-All-Odds"),
    ]

    for candidate in candidates:
        if candidate is not None and candidate.exists():
            return candidate

    if running_in_colab():
        return Path("/content/againstallodds")

    return cwd / "againstallodds"


PROJECT_ROOT = choose_project_root()

CLEAN_NBA_DIR = PROJECT_ROOT / "data" / "clean" / "nba"
CLEAN_NFL_DIR = PROJECT_ROOT / "data" / "clean" / "nfl"
CLEAN_EPL_DIR = PROJECT_ROOT / "data" / "clean" / "soccer"

MODEL_READY_NBA_DIR = PROJECT_ROOT / "data" / "model_ready" / "nba"
MODEL_READY_NFL_DIR = PROJECT_ROOT / "data" / "model_ready" / "nfl"
MODEL_READY_EPL_DIR = PROJECT_ROOT / "data" / "model_ready" / "soccer"

DOCS_DIR = PROJECT_ROOT / "docs"
RESULTS_DIR = PROJECT_ROOT / "results"

for folder in [
    CLEAN_NBA_DIR,
    CLEAN_NFL_DIR,
    CLEAN_EPL_DIR,
    MODEL_READY_NBA_DIR,
    MODEL_READY_NFL_DIR,
    MODEL_READY_EPL_DIR,
    DOCS_DIR,
    RESULTS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"✅ {PROJECT_TITLE} — Week {WEEK_NUMBER}")
print("Project root:", PROJECT_ROOT)

# ------------------------------------------------------------
# 2. Required Week 3–5 input files
# ------------------------------------------------------------

EXPECTED_INPUTS = {
    "nba_games": "nba_games_clean_2021_22_to_2025_26.csv",
    "nba_team_games": "nba_team_game_logs_clean_2021_22_to_2025_26.csv",
    "nfl_games": "nfl_games_clean_2021_to_2025.csv",
    "nfl_team_games": "nfl_team_game_logs_clean_2021_to_2025.csv",
    "epl_games": "premier_league_games_clean_2021_22_to_2025_26.csv",
    "epl_team_games": "premier_league_team_games_clean_2021_22_to_2025_26.csv",
}


def search_locations():
    locations = [
        PROJECT_ROOT,
        PROJECT_ROOT / "data" / "clean",
        Path("/content"),
        Path("/content/week6_inputs"),
        Path.cwd().resolve(),
    ]

    for extra in [
        Path("/content/drive/MyDrive/againstallodds"),
        Path("/content/drive/MyDrive/Against-All-Odds"),
    ]:
        if extra.exists():
            locations.append(extra)

    # Preserve order while removing duplicates.
    return list(dict.fromkeys([p for p in locations if p.exists()]))


def find_exact_file(filename):
    for root in search_locations():
        direct = root / filename
        if direct.exists():
            return direct

    for root in search_locations():
        try:
            matches = list(root.rglob(filename))
        except (PermissionError, OSError):
            matches = []
        if matches:
            return matches[0]

    return None


def resolve_inputs():
    found = {
        key: find_exact_file(filename)
        for key, filename in EXPECTED_INPUTS.items()
    }

    missing = [
        filename
        for key, filename in EXPECTED_INPUTS.items()
        if found[key] is None
    ]

    if missing and running_in_colab():
        print("\nThe following Week 3–5 clean CSV files are missing:")
        for name in missing:
            print(" -", name)

        print(
            "\nUpload the missing CSV files or a ZIP containing them. "
            "Colab will continue after the upload."
        )

        from google.colab import files

        uploaded = files.upload()

        upload_dir = Path("/content/week6_inputs")
        upload_dir.mkdir(parents=True, exist_ok=True)

        for name, blob in uploaded.items():
            destination = upload_dir / Path(name).name
            destination.write_bytes(blob)

            if zipfile.is_zipfile(destination):
                with zipfile.ZipFile(destination, "r") as archive:
                    archive.extractall(upload_dir)

        found = {
            key: find_exact_file(filename)
            for key, filename in EXPECTED_INPUTS.items()
        }

    still_missing = [
        EXPECTED_INPUTS[key]
        for key, path in found.items()
        if path is None
    ]

    if still_missing:
        raise FileNotFoundError(
            "Week 6 cannot begin because these cleaned input files are missing:\n"
            + "\n".join(f" - {name}" for name in still_missing)
            + "\n\nRun Weeks 3–5 or upload their clean CSV outputs first."
        )

    return found


INPUT_PATHS = resolve_inputs()

print("\n✅ Required Week 6 inputs located:")
for key, path in INPUT_PATHS.items():
    print(f" - {key}: {path}")

# ------------------------------------------------------------
# 3. Load clean NBA, NFL, and Premier League data
# ------------------------------------------------------------

nba_games = pd.read_csv(
    INPUT_PATHS["nba_games"],
    dtype={"game_id": "string"},
    parse_dates=["game_date"],
)

nba_team_games = pd.read_csv(
    INPUT_PATHS["nba_team_games"],
    dtype={"game_id": "string"},
    parse_dates=["game_date"],
)

nfl_games = pd.read_csv(
    INPUT_PATHS["nfl_games"],
    dtype={"game_id": "string"},
    parse_dates=["game_date"],
)

nfl_team_games = pd.read_csv(
    INPUT_PATHS["nfl_team_games"],
    dtype={"game_id": "string"},
    parse_dates=["game_date"],
)

epl_games = pd.read_csv(
    INPUT_PATHS["epl_games"],
    dtype={"game_id": "string"},
    parse_dates=["game_date"],
)

epl_team_games = pd.read_csv(
    INPUT_PATHS["epl_team_games"],
    dtype={"game_id": "string"},
    parse_dates=["game_date"],
)


def require_columns(df, required, name):
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(
            f"{name} is missing required columns: {missing}"
        )


require_columns(
    nba_games,
    [
        "game_id",
        "game_date",
        "data_split",
        "home_team_abbr",
        "away_team_abbr",
        "HOME_WIN",
    ],
    "NBA game dataset",
)

require_columns(
    nba_team_games,
    [
        "game_id",
        "game_date",
        "data_split",
        "team_abbreviation",
        "is_home",
        "team_win",
        "pts",
        "opp_pts",
    ],
    "NBA team-game dataset",
)

require_columns(
    nfl_games,
    [
        "game_id",
        "season",
        "game_date",
        "data_split",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
    ],
    "NFL game dataset",
)

require_columns(
    nfl_team_games,
    [
        "game_id",
        "season",
        "game_date",
        "data_split",
        "team_abbr",
        "is_home",
        "team_result",
        "points_for",
        "points_against",
        "point_differential",
    ],
    "NFL team-game dataset",
)

require_columns(
    epl_games,
    [
        "game_id",
        "season",
        "game_date",
        "data_split",
        "home_team",
        "away_team",
        "home_goals",
        "away_goals",
        "HOME_WIN",
    ],
    "Premier League game dataset",
)

require_columns(
    epl_team_games,
    [
        "game_id",
        "season",
        "game_date",
        "data_split",
        "team",
        "is_home",
        "team_result",
        "goals_scored",
        "goals_conceded",
        "goal_difference",
    ],
    "Premier League team-game dataset",
)

print("\n✅ Input schema checks passed")
print("NBA:", nba_games.shape, "| team-game:", nba_team_games.shape)
print("NFL:", nfl_games.shape, "| team-game:", nfl_team_games.shape)
print("EPL:", epl_games.shape, "| team-game:", epl_team_games.shape)

# ------------------------------------------------------------
# 4. General leakage-safe helpers
# ------------------------------------------------------------

def shifted_expanding_mean(series):
    # Current game is removed BEFORE the expanding calculation.
    return series.shift(1).expanding(min_periods=1).mean()


def shifted_rolling_mean(series, window):
    # Current game is removed BEFORE the rolling calculation.
    return (
        series
        .shift(1)
        .rolling(window=window, min_periods=1)
        .mean()
    )


def make_team_history_features(
    df,
    *,
    team_col,
    season_col,
    date_col,
    outcome_col,
    offense_col,
    defense_col,
    differential_col,
    optional_rolling=None,
    points_col=None,
):
    """
    Create team-level pre-game historical features.

    Every expanding/rolling statistic uses shift(1), preventing the
    current game's result or statistics from entering its own predictors.
    """

    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col], errors="coerce")

    numeric_columns = [
        outcome_col,
        offense_col,
        defense_col,
        differential_col,
    ]

    if points_col is not None and points_col in out.columns:
        numeric_columns.append(points_col)

    for col in numeric_columns:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out = (
        out
        .sort_values(
            [team_col, season_col, date_col, "game_id"],
            kind="stable",
        )
        .reset_index(drop=True)
    )

    group_keys = [team_col, season_col]

    out["GAMES_BEFORE"] = (
        out
        .groupby(group_keys, sort=False)
        .cumcount()
    )

    # Season-to-date win percentage before the current game.
    out["WIN_PCT_PRE"] = (
        out
        .groupby(group_keys, sort=False)[outcome_col]
        .transform(shifted_expanding_mean)
    )

    # Recent form.
    for window in [3, 5, 10]:
        out[f"FORM_{window}"] = (
            out
            .groupby(group_keys, sort=False)[outcome_col]
            .transform(
                lambda s, w=window: shifted_rolling_mean(s, w)
            )
        )

    # Team strength via scoring/goal differential.
    out["POINT_GOAL_DIFF_PRE"] = (
        out
        .groupby(group_keys, sort=False)[differential_col]
        .transform(shifted_expanding_mean)
    )

    # Basic offense and defense.
    out["OFFENSE_PRE"] = (
        out
        .groupby(group_keys, sort=False)[offense_col]
        .transform(shifted_expanding_mean)
    )

    out["DEFENSE_ALLOWED_PRE"] = (
        out
        .groupby(group_keys, sort=False)[defense_col]
        .transform(shifted_expanding_mean)
    )

    # Recent offense and defense.
    out["OFFENSE_LAST5"] = (
        out
        .groupby(group_keys, sort=False)[offense_col]
        .transform(lambda s: shifted_rolling_mean(s, 5))
    )

    out["DEFENSE_LAST5"] = (
        out
        .groupby(group_keys, sort=False)[defense_col]
        .transform(lambda s: shifted_rolling_mean(s, 5))
    )

    out["DIFF_LAST5"] = (
        out
        .groupby(group_keys, sort=False)[differential_col]
        .transform(lambda s: shifted_rolling_mean(s, 5))
    )

    # Rest: current game date minus the previous game date.
    out["REST_DAYS"] = (
        out
        .groupby(group_keys, sort=False)[date_col]
        .diff()
        .dt.days
    )

    if points_col is not None and points_col in out.columns:
        out["LEAGUE_POINTS_PER_GAME_PRE"] = (
            out
            .groupby(group_keys, sort=False)[points_col]
            .transform(shifted_expanding_mean)
        )

    # Sport-specific optional rolling features.
    optional_rolling = optional_rolling or {}

    for source_col, output_name in optional_rolling.items():
        if source_col not in out.columns:
            continue

        out[source_col] = pd.to_numeric(
            out[source_col],
            errors="coerce",
        )

        out[output_name] = (
            out
            .groupby(group_keys, sort=False)[source_col]
            .transform(lambda s: shifted_rolling_mean(s, 5))
        )

    return out


def elo_expected(rating_a, rating_b):
    return 1.0 / (
        1.0 + 10.0 ** ((rating_b - rating_a) / 400.0)
    )


def add_pre_game_elo(
    games,
    *,
    home_team_col,
    away_team_col,
    actual_home_col,
    sport,
):
    """
    Build one chronological Elo sequence.

    The ratings saved on each row are the ratings BEFORE that game.
    Ratings then update only after the actual outcome is observed.
    """

    out = (
        games
        .copy()
        .sort_values(["game_date", "game_id"], kind="stable")
        .reset_index(drop=True)
    )

    ratings = {}
    home_pre = []
    away_pre = []

    k_factor = ELO_K[sport]

    for row in out.itertuples(index=False):
        home_team = getattr(row, home_team_col)
        away_team = getattr(row, away_team_col)
        actual_home = float(getattr(row, actual_home_col))

        home_rating = ratings.get(home_team, ELO_INITIAL)
        away_rating = ratings.get(away_team, ELO_INITIAL)

        home_pre.append(home_rating)
        away_pre.append(away_rating)

        expected_home = elo_expected(
            home_rating,
            away_rating,
        )

        ratings[home_team] = (
            home_rating
            + k_factor * (actual_home - expected_home)
        )

        ratings[away_team] = (
            away_rating
            + k_factor
            * (
                (1.0 - actual_home)
                - (1.0 - expected_home)
            )
        )

    out["HOME_ELO_PRE"] = home_pre
    out["AWAY_ELO_PRE"] = away_pre
    out["ELO_DIFF"] = (
        out["HOME_ELO_PRE"]
        - out["AWAY_ELO_PRE"]
    )

    return out


def attach_opponent_elo_and_schedule_strength(
    team_history,
    elo_games,
    *,
    team_col,
    season_col,
    home_team_col,
    away_team_col,
):
    """
    Add the current opponent's pre-game Elo and a leakage-safe rolling
    average of prior opponents' Elo (recent schedule strength).
    """

    out = team_history.copy()

    home_map = (
        elo_games[
            [
                "game_id",
                home_team_col,
                "AWAY_ELO_PRE",
            ]
        ]
        .rename(
            columns={
                home_team_col: team_col,
                "AWAY_ELO_PRE": "OPPONENT_ELO_PRE",
            }
        )
    )

    away_map = (
        elo_games[
            [
                "game_id",
                away_team_col,
                "HOME_ELO_PRE",
            ]
        ]
        .rename(
            columns={
                away_team_col: team_col,
                "HOME_ELO_PRE": "OPPONENT_ELO_PRE",
            }
        )
    )

    opponent_map = pd.concat(
        [home_map, away_map],
        ignore_index=True,
    )

    out = out.merge(
        opponent_map,
        on=["game_id", team_col],
        how="left",
        validate="one_to_one",
    )

    out = (
        out
        .sort_values(
            [team_col, season_col, "game_date", "game_id"],
            kind="stable",
        )
        .reset_index(drop=True)
    )

    out["SCHEDULE_STRENGTH_LAST5"] = (
        out
        .groupby([team_col, season_col], sort=False)[
            "OPPONENT_ELO_PRE"
        ]
        .transform(lambda s: shifted_rolling_mean(s, 5))
    )

    return out


def verify_shift_rule(
    team_df,
    *,
    team_col,
    season_col,
    outcome_col,
):
    """
    Manually test several team-seasons to confirm FORM_3 is based
    on prior completed games and excludes the current game.
    """

    checked = 0

    for _, group in team_df.groupby(
        [team_col, season_col],
        sort=False,
    ):
        group = (
            group
            .sort_values(["game_date", "game_id"])
            .reset_index(drop=True)
        )

        if len(group) < 4:
            continue

        position = 3

        expected = (
            group
            .loc[max(0, position - 3):position - 1, outcome_col]
            .mean()
        )

        actual = group.loc[position, "FORM_3"]

        if not np.isclose(
            float(actual),
            float(expected),
            equal_nan=False,
        ):
            raise AssertionError(
                "Leakage audit failed: FORM_3 did not equal "
                "the previous three completed results."
            )

        checked += 1

        if checked >= 10:
            break

    if checked == 0:
        raise AssertionError(
            "Leakage audit could not find a team-season "
            "with at least four games."
        )

    return True


def merge_team_features_to_games(
    games,
    team_history,
    *,
    home_team_col,
    away_team_col,
    team_col,
    feature_columns,
):
    home = (
        team_history[
            team_history["is_home"].eq(1)
        ][["game_id", team_col] + feature_columns]
        .copy()
    )

    away = (
        team_history[
            team_history["is_home"].eq(0)
        ][["game_id", team_col] + feature_columns]
        .copy()
    )

    home = home.rename(
        columns={
            team_col: "_HOME_TEAM_CHECK",
            **{
                feature: f"HOME_{feature}"
                for feature in feature_columns
            },
        }
    )

    away = away.rename(
        columns={
            team_col: "_AWAY_TEAM_CHECK",
            **{
                feature: f"AWAY_{feature}"
                for feature in feature_columns
            },
        }
    )

    merged = (
        games
        .merge(
            home,
            on="game_id",
            how="left",
            validate="one_to_one",
        )
        .merge(
            away,
            on="game_id",
            how="left",
            validate="one_to_one",
        )
    )

    home_match = merged["_HOME_TEAM_CHECK"].eq(
        merged[home_team_col]
    )

    away_match = merged["_AWAY_TEAM_CHECK"].eq(
        merged[away_team_col]
    )

    if not home_match.all():
        raise AssertionError(
            "Home-team feature merge validation failed."
        )

    if not away_match.all():
        raise AssertionError(
            "Away-team feature merge validation failed."
        )

    return merged.drop(
        columns=[
            "_HOME_TEAM_CHECK",
            "_AWAY_TEAM_CHECK",
        ]
    )


CORE_TEAM_FEATURES = [
    "GAMES_BEFORE",
    "WIN_PCT_PRE",
    "FORM_3",
    "FORM_5",
    "FORM_10",
    "POINT_GOAL_DIFF_PRE",
    "OFFENSE_PRE",
    "DEFENSE_ALLOWED_PRE",
    "OFFENSE_LAST5",
    "DEFENSE_LAST5",
    "DIFF_LAST5",
    "REST_DAYS",
    "OPPONENT_ELO_PRE",
    "SCHEDULE_STRENGTH_LAST5",
]


def add_matchup_differences(df):
    out = df.copy()

    out["WIN_PCT_DIFF"] = (
        out["HOME_WIN_PCT_PRE"]
        - out["AWAY_WIN_PCT_PRE"]
    )

    out["FORM_3_DIFF"] = (
        out["HOME_FORM_3"]
        - out["AWAY_FORM_3"]
    )

    out["FORM_5_DIFF"] = (
        out["HOME_FORM_5"]
        - out["AWAY_FORM_5"]
    )

    out["FORM_10_DIFF"] = (
        out["HOME_FORM_10"]
        - out["AWAY_FORM_10"]
    )

    # FORM_DIFF is the main recent-form difference for later models.
    out["FORM_DIFF"] = out["FORM_5_DIFF"]

    out["POINT_GOAL_DIFF"] = (
        out["HOME_POINT_GOAL_DIFF_PRE"]
        - out["AWAY_POINT_GOAL_DIFF_PRE"]
    )

    out["OFFENSE_DIFF"] = (
        out["HOME_OFFENSE_PRE"]
        - out["AWAY_OFFENSE_PRE"]
    )

    # Lower points/goals allowed means better defense.
    # Positive DEFENSE_DIFF therefore favors the home team.
    out["DEFENSE_DIFF"] = (
        out["AWAY_DEFENSE_ALLOWED_PRE"]
        - out["HOME_DEFENSE_ALLOWED_PRE"]
    )

    out["RECENT_OFFENSE_DIFF"] = (
        out["HOME_OFFENSE_LAST5"]
        - out["AWAY_OFFENSE_LAST5"]
    )

    out["RECENT_DEFENSE_DIFF"] = (
        out["AWAY_DEFENSE_LAST5"]
        - out["HOME_DEFENSE_LAST5"]
    )

    out["REST_DIFF"] = (
        out["HOME_REST_DAYS"]
        - out["AWAY_REST_DAYS"]
    )

    out["SCHEDULE_STRENGTH_DIFF"] = (
        out["HOME_SCHEDULE_STRENGTH_LAST5"]
        - out["AWAY_SCHEDULE_STRENGTH_LAST5"]
    )

    # Each final game row is represented from the home-team perspective.
    out["HOME_ADVANTAGE"] = 1

    return out


def add_optional_matchup_differences(df):
    """
    Add sport-specific differences only if those optional source
    statistics exist in the clean data.
    """

    out = df.copy()

    direct_pairs = {
        # NBA
        "OFF_RATING_DIFF": (
            "HOME_OFF_RATING_LAST5",
            "AWAY_OFF_RATING_LAST5",
        ),
        "NET_RATING_DIFF": (
            "HOME_NET_RATING_LAST5",
            "AWAY_NET_RATING_LAST5",
        ),
        "EFG_DIFF": (
            "HOME_EFG_LAST5",
            "AWAY_EFG_LAST5",
        ),
        "THREE_PT_DIFF": (
            "HOME_THREE_PT_LAST5",
            "AWAY_THREE_PT_LAST5",
        ),

        # Soccer
        "SHOTS_DIFF": (
            "HOME_SHOTS_LAST5",
            "AWAY_SHOTS_LAST5",
        ),
        "SHOTS_ON_TARGET_DIFF": (
            "HOME_SHOTS_ON_TARGET_LAST5",
            "AWAY_SHOTS_ON_TARGET_LAST5",
        ),
    }

    reverse_pairs = {
        # Lower defensive rating is better.
        "DEF_RATING_DIFF": (
            "AWAY_DEF_RATING_LAST5",
            "HOME_DEF_RATING_LAST5",
        ),
    }

    for output, (left, right) in direct_pairs.items():
        if left in out.columns and right in out.columns:
            out[output] = out[left] - out[right]

    for output, (left, right) in reverse_pairs.items():
        if left in out.columns and right in out.columns:
            out[output] = out[left] - out[right]

    return out


def finalize_model_ready(
    df,
    *,
    sport,
    target_col,
    identifier_columns,
    optional_columns=None,
):
    """
    Keep identifiers, target, and pre-game features only.

    Post-game score and box-score fields are intentionally excluded.
    """

    out = df.copy()

    required_features = [
        "HOME_ELO_PRE",
        "AWAY_ELO_PRE",
        "ELO_DIFF",
        "HOME_WIN_PCT_PRE",
        "AWAY_WIN_PCT_PRE",
        "WIN_PCT_DIFF",
        "HOME_FORM_3",
        "AWAY_FORM_3",
        "FORM_3_DIFF",
        "HOME_FORM_5",
        "AWAY_FORM_5",
        "FORM_DIFF",
        "HOME_FORM_10",
        "AWAY_FORM_10",
        "FORM_10_DIFF",
        "HOME_POINT_GOAL_DIFF_PRE",
        "AWAY_POINT_GOAL_DIFF_PRE",
        "POINT_GOAL_DIFF",
        "HOME_OFFENSE_PRE",
        "AWAY_OFFENSE_PRE",
        "OFFENSE_DIFF",
        "HOME_DEFENSE_ALLOWED_PRE",
        "AWAY_DEFENSE_ALLOWED_PRE",
        "DEFENSE_DIFF",
        "HOME_REST_DAYS",
        "AWAY_REST_DAYS",
        "REST_DIFF",
        "HOME_OPPONENT_ELO_PRE",
        "AWAY_OPPONENT_ELO_PRE",
        "HOME_SCHEDULE_STRENGTH_LAST5",
        "AWAY_SCHEDULE_STRENGTH_LAST5",
        "SCHEDULE_STRENGTH_DIFF",
        "HOME_ADVANTAGE",
    ]

    optional_columns = optional_columns or []

    optional_available = [
        col
        for col in optional_columns
        if col in out.columns
    ]

    before = len(out)

    # Remove rows where a team's prior history is not yet available.
    out = out.dropna(
        subset=required_features + [target_col]
    ).copy()

    # Final target must be binary.
    target_numeric = pd.to_numeric(
        out[target_col],
        errors="coerce",
    )

    out = out[target_numeric.isin([0, 1])].copy()

    out[target_col] = pd.to_numeric(
        out[target_col],
        errors="raise",
    ).astype("int8")

    keep = (
        identifier_columns
        + [target_col]
        + required_features
        + optional_available
    )

    keep = list(dict.fromkeys(keep))

    out = out[keep].copy()

    out = (
        out
        .sort_values(["game_date", "game_id"], kind="stable")
        .reset_index(drop=True)
    )

    if out["game_id"].duplicated().any():
        raise AssertionError(
            f"{sport}: duplicate game_id values remain."
        )

    if not out["game_date"].is_monotonic_increasing:
        raise AssertionError(
            f"{sport}: model-ready games are not chronological."
        )

    if out[required_features].isna().any().any():
        raise AssertionError(
            f"{sport}: required model-ready features contain missing values."
        )

    # Explicit predictor-leakage audit.
    forbidden_predictor_terms = [
        "home_score",
        "away_score",
        "home_goals",
        "away_goals",
        "home_pts",
        "away_pts",
        "home_wl",
        "away_wl",
        "game_total_points",
        "home_margin",
        "current_game",
    ]

    leaked = [
        col
        for col in out.columns
        if any(
            term in col.lower()
            for term in forbidden_predictor_terms
        )
        and col not in identifier_columns
        and col != target_col
    ]

    if leaked:
        raise AssertionError(
            f"{sport}: post-game predictor columns found: {leaked}"
        )

    print(
        f"✅ {sport}: {before:,} clean games -> "
        f"{len(out):,} model-ready games"
    )

    return out, required_features, optional_available


# ------------------------------------------------------------
# 5. NBA feature engineering
# ------------------------------------------------------------

nba_team = nba_team_games.copy()

nba_team["team_abbreviation"] = (
    nba_team["team_abbreviation"]
    .astype("string")
    .str.strip()
)

if "season_start" in nba_team.columns:
    NBA_SEASON_COL = "season_start"
elif "season_label" in nba_team.columns:
    NBA_SEASON_COL = "season_label"
else:
    raise ValueError(
        "NBA team-game data require season_start or season_label."
    )

nba_team["team_win"] = pd.to_numeric(
    nba_team["team_win"],
    errors="coerce",
)

nba_team["pts"] = pd.to_numeric(
    nba_team["pts"],
    errors="coerce",
)

nba_team["opp_pts"] = pd.to_numeric(
    nba_team["opp_pts"],
    errors="coerce",
)

if "plus_minus" in nba_team.columns:
    nba_team["plus_minus"] = pd.to_numeric(
        nba_team["plus_minus"],
        errors="coerce",
    )
    nba_team["NBA_DIFF_VALUE"] = nba_team["plus_minus"]
else:
    nba_team["NBA_DIFF_VALUE"] = (
        nba_team["pts"]
        - nba_team["opp_pts"]
    )

nba_optional_rolling = {
    "off_rating": "OFF_RATING_LAST5",
    "def_rating": "DEF_RATING_LAST5",
    "net_rating": "NET_RATING_LAST5",
    "efg_pct": "EFG_LAST5",
    "fg3_pct": "THREE_PT_LAST5",
}

nba_history = make_team_history_features(
    nba_team,
    team_col="team_abbreviation",
    season_col=NBA_SEASON_COL,
    date_col="game_date",
    outcome_col="team_win",
    offense_col="pts",
    defense_col="opp_pts",
    differential_col="NBA_DIFF_VALUE",
    optional_rolling=nba_optional_rolling,
)

verify_shift_rule(
    nba_history,
    team_col="team_abbreviation",
    season_col=NBA_SEASON_COL,
    outcome_col="team_win",
)

print("\n✅ NBA shift(1) leakage audit passed")

nba_games_features = nba_games.copy()

nba_games_features["NBA_ELO_ACTUAL_HOME"] = pd.to_numeric(
    nba_games_features["HOME_WIN"],
    errors="coerce",
)

nba_games_features = add_pre_game_elo(
    nba_games_features,
    home_team_col="home_team_abbr",
    away_team_col="away_team_abbr",
    actual_home_col="NBA_ELO_ACTUAL_HOME",
    sport="NBA",
)

nba_history = attach_opponent_elo_and_schedule_strength(
    nba_history,
    nba_games_features,
    team_col="team_abbreviation",
    season_col=NBA_SEASON_COL,
    home_team_col="home_team_abbr",
    away_team_col="away_team_abbr",
)

nba_team_feature_cols = CORE_TEAM_FEATURES + [
    col
    for col in nba_optional_rolling.values()
    if col in nba_history.columns
]

nba_games_features = merge_team_features_to_games(
    nba_games_features,
    nba_history,
    home_team_col="home_team_abbr",
    away_team_col="away_team_abbr",
    team_col="team_abbreviation",
    feature_columns=nba_team_feature_cols,
)

nba_games_features = add_matchup_differences(
    nba_games_features
)

nba_games_features = add_optional_matchup_differences(
    nba_games_features
)

nba_identifier_columns = [
    col
    for col in [
        "game_id",
        "game_date",
        "season_start",
        "season_end",
        "season_label",
        "data_split",
        "home_team_abbr",
        "home_team_name",
        "away_team_abbr",
        "away_team_name",
    ]
    if col in nba_games_features.columns
]

nba_optional_differences = [
    "OFF_RATING_DIFF",
    "DEF_RATING_DIFF",
    "NET_RATING_DIFF",
    "EFG_DIFF",
    "THREE_PT_DIFF",
]

nba_model_ready, nba_required_features, nba_optional_features = (
    finalize_model_ready(
        nba_games_features,
        sport="NBA",
        target_col="HOME_WIN",
        identifier_columns=nba_identifier_columns,
        optional_columns=nba_optional_differences,
    )
)

print("\nNBA model-ready preview:")
display(nba_model_ready.head())


# ------------------------------------------------------------
# 6. NFL feature engineering
# ------------------------------------------------------------

nfl_team = nfl_team_games.copy()

nfl_team["team_abbr"] = (
    nfl_team["team_abbr"]
    .astype("string")
    .str.strip()
)

# Use 1 for win, 0.5 for tie, 0 for loss when updating
# historical team strength.
nfl_team["NFL_RESULT_VALUE"] = (
    nfl_team["team_result"]
    .astype("string")
    .str.upper()
    .map({
        "W": 1.0,
        "T": 0.5,
        "L": 0.0,
    })
)

nfl_history = make_team_history_features(
    nfl_team,
    team_col="team_abbr",
    season_col="season",
    date_col="game_date",
    outcome_col="NFL_RESULT_VALUE",
    offense_col="points_for",
    defense_col="points_against",
    differential_col="point_differential",
)

verify_shift_rule(
    nfl_history,
    team_col="team_abbr",
    season_col="season",
    outcome_col="NFL_RESULT_VALUE",
)

print("\n✅ NFL shift(1) leakage audit passed")

nfl_games_features = nfl_games.copy()

nfl_home_score = pd.to_numeric(
    nfl_games_features["home_score"],
    errors="coerce",
)

nfl_away_score = pd.to_numeric(
    nfl_games_features["away_score"],
    errors="coerce",
)

# Elo: win = 1, tie = 0.5, loss = 0.
nfl_games_features["NFL_ELO_ACTUAL_HOME"] = np.select(
    [
        nfl_home_score.gt(nfl_away_score),
        nfl_home_score.lt(nfl_away_score),
    ],
    [1.0, 0.0],
    default=0.5,
)

# Reconstruct HOME_WIN if needed.
if "HOME_WIN" not in nfl_games_features.columns:
    nfl_games_features["HOME_WIN"] = np.select(
        [
            nfl_home_score.gt(nfl_away_score),
            nfl_home_score.lt(nfl_away_score),
        ],
        [1.0, 0.0],
        default=np.nan,
    )

nfl_games_features = add_pre_game_elo(
    nfl_games_features,
    home_team_col="home_team",
    away_team_col="away_team",
    actual_home_col="NFL_ELO_ACTUAL_HOME",
    sport="NFL",
)

nfl_history = attach_opponent_elo_and_schedule_strength(
    nfl_history,
    nfl_games_features,
    team_col="team_abbr",
    season_col="season",
    home_team_col="home_team",
    away_team_col="away_team",
)

nfl_games_features = merge_team_features_to_games(
    nfl_games_features,
    nfl_history,
    home_team_col="home_team",
    away_team_col="away_team",
    team_col="team_abbr",
    feature_columns=CORE_TEAM_FEATURES,
)

nfl_games_features = add_matchup_differences(
    nfl_games_features
)

nfl_identifier_columns = [
    col
    for col in [
        "game_id",
        "game_date",
        "season",
        "week",
        "data_split",
        "home_team",
        "away_team",
    ]
    if col in nfl_games_features.columns
]

nfl_model_ready, nfl_required_features, nfl_optional_features = (
    finalize_model_ready(
        nfl_games_features,
        sport="NFL",
        target_col="HOME_WIN",
        identifier_columns=nfl_identifier_columns,
    )
)

print(
    "\nNFL note: ties update historical Elo and team strength "
    "using 0.5, but tied games are excluded from the final "
    "binary HOME_WIN model-ready dataset."
)

print("\nNFL model-ready preview:")
display(nfl_model_ready.head())


# ------------------------------------------------------------
# 7. Premier League feature engineering
# ------------------------------------------------------------

epl_team = epl_team_games.copy()

epl_team["team"] = (
    epl_team["team"]
    .astype("string")
    .str.strip()
)

# For the project's binary outcome/form definition:
# win = 1; draw or loss = 0.
epl_team["EPL_WIN_VALUE"] = (
    epl_team["team_result"]
    .astype("string")
    .str.upper()
    .eq("W")
    .astype(float)
)

epl_optional_rolling = {
    "shots": "SHOTS_LAST5",
    "shots_on_target": "SHOTS_ON_TARGET_LAST5",
}

epl_history = make_team_history_features(
    epl_team,
    team_col="team",
    season_col="season",
    date_col="game_date",
    outcome_col="EPL_WIN_VALUE",
    offense_col="goals_scored",
    defense_col="goals_conceded",
    differential_col="goal_difference",
    optional_rolling=epl_optional_rolling,
    points_col=(
        "points_earned"
        if "points_earned" in epl_team.columns
        else None
    ),
)

verify_shift_rule(
    epl_history,
    team_col="team",
    season_col="season",
    outcome_col="EPL_WIN_VALUE",
)

print("\n✅ Premier League shift(1) leakage audit passed")

epl_games_features = epl_games.copy()

epl_home_goals = pd.to_numeric(
    epl_games_features["home_goals"],
    errors="coerce",
)

epl_away_goals = pd.to_numeric(
    epl_games_features["away_goals"],
    errors="coerce",
)

# Elo: home win = 1, draw = 0.5, away win = 0.
epl_games_features["EPL_ELO_ACTUAL_HOME"] = np.select(
    [
        epl_home_goals.gt(epl_away_goals),
        epl_home_goals.lt(epl_away_goals),
    ],
    [1.0, 0.0],
    default=0.5,
)

epl_games_features = add_pre_game_elo(
    epl_games_features,
    home_team_col="home_team",
    away_team_col="away_team",
    actual_home_col="EPL_ELO_ACTUAL_HOME",
    sport="Premier League",
)

epl_history = attach_opponent_elo_and_schedule_strength(
    epl_history,
    epl_games_features,
    team_col="team",
    season_col="season",
    home_team_col="home_team",
    away_team_col="away_team",
)

epl_team_feature_cols = CORE_TEAM_FEATURES.copy()

if "LEAGUE_POINTS_PER_GAME_PRE" in epl_history.columns:
    epl_team_feature_cols.append(
        "LEAGUE_POINTS_PER_GAME_PRE"
    )

epl_team_feature_cols += [
    col
    for col in epl_optional_rolling.values()
    if col in epl_history.columns
]

epl_games_features = merge_team_features_to_games(
    epl_games_features,
    epl_history,
    home_team_col="home_team",
    away_team_col="away_team",
    team_col="team",
    feature_columns=epl_team_feature_cols,
)

epl_games_features = add_matchup_differences(
    epl_games_features
)

if (
    "HOME_LEAGUE_POINTS_PER_GAME_PRE"
    in epl_games_features.columns
    and "AWAY_LEAGUE_POINTS_PER_GAME_PRE"
    in epl_games_features.columns
):
    epl_games_features["LEAGUE_POINTS_RATE_DIFF"] = (
        epl_games_features["HOME_LEAGUE_POINTS_PER_GAME_PRE"]
        - epl_games_features["AWAY_LEAGUE_POINTS_PER_GAME_PRE"]
    )

epl_games_features = add_optional_matchup_differences(
    epl_games_features
)

epl_identifier_columns = [
    col
    for col in [
        "game_id",
        "game_date",
        "season",
        "season_code",
        "data_split",
        "home_team",
        "away_team",
    ]
    if col in epl_games_features.columns
]

epl_optional_differences = [
    "LEAGUE_POINTS_RATE_DIFF",
    "SHOTS_DIFF",
    "SHOTS_ON_TARGET_DIFF",
]

epl_model_ready, epl_required_features, epl_optional_features = (
    finalize_model_ready(
        epl_games_features,
        sport="Premier League",
        target_col="HOME_WIN",
        identifier_columns=epl_identifier_columns,
        optional_columns=epl_optional_differences,
    )
)

print("\nPremier League model-ready preview:")
display(epl_model_ready.head())


# ------------------------------------------------------------
# 8. Cross-sport Week 6 quality checks
# ------------------------------------------------------------

def quality_row(
    sport,
    source_games,
    model_ready,
    required_features,
):
    return {
        "sport": sport,
        "clean_input_games": int(len(source_games)),
        "model_ready_games": int(len(model_ready)),
        "rows_removed_for_missing_prior_history_or_ties": int(
            len(source_games) - len(model_ready)
        ),
        "unique_game_ids": bool(
            model_ready["game_id"].is_unique
        ),
        "chronological_order": bool(
            model_ready["game_date"].is_monotonic_increasing
        ),
        "binary_HOME_WIN": bool(
            model_ready["HOME_WIN"]
            .isin([0, 1])
            .all()
        ),
        "required_features_complete": bool(
            not model_ready[
                required_features
            ].isna().any().any()
        ),
        "shift_1_rule_used": True,
    }


quality_report = pd.DataFrame([
    quality_row(
        "NBA",
        nba_games,
        nba_model_ready,
        nba_required_features,
    ),
    quality_row(
        "NFL",
        nfl_games,
        nfl_model_ready,
        nfl_required_features,
    ),
    quality_row(
        "Premier League",
        epl_games,
        epl_model_ready,
        epl_required_features,
    ),
])

quality_report["status"] = np.where(
    quality_report[
        [
            "unique_game_ids",
            "chronological_order",
            "binary_HOME_WIN",
            "required_features_complete",
            "shift_1_rule_used",
        ]
    ].all(axis=1),
    "PASS",
    "FAIL",
)

print("\nWeek 6 quality report:")
display(quality_report)

if not quality_report["status"].eq("PASS").all():
    raise RuntimeError(
        "Week 6 quality checks failed. "
        "Review the table above before saving."
    )

# Model-ready split summary.
def split_summary(df, sport):
    if "data_split" not in df.columns:
        return pd.DataFrame()

    return (
        df
        .groupby("data_split", dropna=False)
        .agg(
            games=("game_id", "nunique"),
            home_win_rate=("HOME_WIN", "mean"),
        )
        .reset_index()
        .assign(sport=sport)
        [["sport", "data_split", "games", "home_win_rate"]]
    )


split_summary_df = pd.concat(
    [
        split_summary(nba_model_ready, "NBA"),
        split_summary(nfl_model_ready, "NFL"),
        split_summary(epl_model_ready, "Premier League"),
    ],
    ignore_index=True,
)

print("\nModel-ready split summary:")
display(split_summary_df)

# Missingness report.
def feature_missingness(df, sport):
    feature_cols = [
        col
        for col in df.columns
        if col not in {
            "game_id",
            "game_date",
            "HOME_WIN",
            "data_split",
            "season",
            "season_start",
            "season_end",
            "season_label",
            "season_code",
            "week",
            "home_team",
            "away_team",
            "home_team_abbr",
            "away_team_abbr",
            "home_team_name",
            "away_team_name",
        }
    ]

    return pd.DataFrame({
        "sport": sport,
        "feature": feature_cols,
        "missing_count": [
            int(df[col].isna().sum())
            for col in feature_cols
        ],
        "missing_pct": [
            round(float(df[col].isna().mean() * 100), 4)
            for col in feature_cols
        ],
    })


missingness_report = pd.concat(
    [
        feature_missingness(nba_model_ready, "NBA"),
        feature_missingness(nfl_model_ready, "NFL"),
        feature_missingness(epl_model_ready, "Premier League"),
    ],
    ignore_index=True,
)

# ------------------------------------------------------------
# 9. Save the three Week 6 model-ready datasets
# ------------------------------------------------------------

NBA_OUTPUT = (
    MODEL_READY_NBA_DIR
    / "nba_model_ready_2021_22_to_2025_26.csv"
)

NFL_OUTPUT = (
    MODEL_READY_NFL_DIR
    / "nfl_model_ready_2021_to_2025.csv"
)

EPL_OUTPUT = (
    MODEL_READY_EPL_DIR
    / "premier_league_model_ready_2021_22_to_2025_26.csv"
)

nba_model_ready.to_csv(
    NBA_OUTPUT,
    index=False,
)

nfl_model_ready.to_csv(
    NFL_OUTPUT,
    index=False,
)

epl_model_ready.to_csv(
    EPL_OUTPUT,
    index=False,
)

QUALITY_OUTPUT = (
    RESULTS_DIR
    / "week6_feature_engineering_quality_report.csv"
)

SPLIT_OUTPUT = (
    RESULTS_DIR
    / "week6_model_ready_split_summary.csv"
)

MISSINGNESS_OUTPUT = (
    RESULTS_DIR
    / "week6_model_ready_missingness.csv"
)

quality_report.to_csv(
    QUALITY_OUTPUT,
    index=False,
)

split_summary_df.to_csv(
    SPLIT_OUTPUT,
    index=False,
)

missingness_report.to_csv(
    MISSINGNESS_OUTPUT,
    index=False,
)

print("\n✅ Week 6 datasets and reports saved")
print(" -", NBA_OUTPUT)
print(" -", NFL_OUTPUT)
print(" -", EPL_OUTPUT)
print(" -", QUALITY_OUTPUT)
print(" -", SPLIT_OUTPUT)
print(" -", MISSINGNESS_OUTPUT)

# ------------------------------------------------------------
# 10. Week 6 feature dictionary
# ------------------------------------------------------------

feature_dictionary = pd.DataFrame([
    ["HOME_ELO_PRE", "ALL", "Home team's Elo before the current game", "Chronological Elo; rating stored before current-game update"],
    ["AWAY_ELO_PRE", "ALL", "Away team's Elo before the current game", "Chronological Elo; rating stored before current-game update"],
    ["ELO_DIFF", "ALL", "Home Elo minus away Elo", "HOME_ELO_PRE - AWAY_ELO_PRE"],
    ["HOME_WIN_PCT_PRE", "ALL", "Home season-to-date result rate", "Previous games only; shift(1) before expanding mean"],
    ["AWAY_WIN_PCT_PRE", "ALL", "Away season-to-date result rate", "Previous games only; shift(1) before expanding mean"],
    ["WIN_PCT_DIFF", "ALL", "Home minus away win percentage", "HOME_WIN_PCT_PRE - AWAY_WIN_PCT_PRE"],
    ["HOME_FORM_3", "ALL", "Home recent form over up to the previous 3 games", "shift(1).rolling(3)"],
    ["AWAY_FORM_3", "ALL", "Away recent form over up to the previous 3 games", "shift(1).rolling(3)"],
    ["FORM_3_DIFF", "ALL", "Home minus away last-3 form", "HOME_FORM_3 - AWAY_FORM_3"],
    ["HOME_FORM_5", "ALL", "Home recent form over up to the previous 5 games", "shift(1).rolling(5)"],
    ["AWAY_FORM_5", "ALL", "Away recent form over up to the previous 5 games", "shift(1).rolling(5)"],
    ["FORM_DIFF", "ALL", "Primary recent-form difference", "HOME_FORM_5 - AWAY_FORM_5"],
    ["HOME_FORM_10", "ALL", "Home recent form over up to the previous 10 games", "shift(1).rolling(10)"],
    ["AWAY_FORM_10", "ALL", "Away recent form over up to the previous 10 games", "shift(1).rolling(10)"],
    ["FORM_10_DIFF", "ALL", "Home minus away last-10 form", "HOME_FORM_10 - AWAY_FORM_10"],
    ["HOME_POINT_GOAL_DIFF_PRE", "ALL", "Home historical average point/goal differential", "Previous games only"],
    ["AWAY_POINT_GOAL_DIFF_PRE", "ALL", "Away historical average point/goal differential", "Previous games only"],
    ["POINT_GOAL_DIFF", "ALL", "Home minus away point/goal-differential strength", "Home - Away"],
    ["HOME_OFFENSE_PRE", "ALL", "Home historical average points/goals scored", "Previous games only"],
    ["AWAY_OFFENSE_PRE", "ALL", "Away historical average points/goals scored", "Previous games only"],
    ["OFFENSE_DIFF", "ALL", "Home minus away offensive production", "Home - Away"],
    ["HOME_DEFENSE_ALLOWED_PRE", "ALL", "Home historical average points/goals allowed", "Previous games only"],
    ["AWAY_DEFENSE_ALLOWED_PRE", "ALL", "Away historical average points/goals allowed", "Previous games only"],
    ["DEFENSE_DIFF", "ALL", "Defensive advantage from home perspective", "Away allowed - Home allowed"],
    ["HOME_REST_DAYS", "ALL", "Days since home team's previous game", "Current date - previous completed game date"],
    ["AWAY_REST_DAYS", "ALL", "Days since away team's previous game", "Current date - previous completed game date"],
    ["REST_DIFF", "ALL", "Home rest minus away rest", "HOME_REST_DAYS - AWAY_REST_DAYS"],
    ["HOME_OPPONENT_ELO_PRE", "ALL", "Current opponent strength faced by home team", "Away team's pre-game Elo"],
    ["AWAY_OPPONENT_ELO_PRE", "ALL", "Current opponent strength faced by away team", "Home team's pre-game Elo"],
    ["HOME_SCHEDULE_STRENGTH_LAST5", "ALL", "Average Elo of home team's prior opponents", "shift(1).rolling(5) on opponent pre-game Elo"],
    ["AWAY_SCHEDULE_STRENGTH_LAST5", "ALL", "Average Elo of away team's prior opponents", "shift(1).rolling(5) on opponent pre-game Elo"],
    ["SCHEDULE_STRENGTH_DIFF", "ALL", "Home minus away recent schedule strength", "Home - Away"],
    ["HOME_ADVANTAGE", "ALL", "Home-team context indicator", "1 for home-perspective game rows"],
    ["OFF_RATING_DIFF", "NBA", "Difference in rolling offensive rating where available", "Historical rolling feature only"],
    ["DEF_RATING_DIFF", "NBA", "Home defensive-rating advantage where available", "Historical rolling feature only"],
    ["NET_RATING_DIFF", "NBA", "Difference in rolling net rating where available", "Historical rolling feature only"],
    ["EFG_DIFF", "NBA", "Difference in rolling effective field-goal percentage", "Historical rolling feature only"],
    ["THREE_PT_DIFF", "NBA", "Difference in rolling three-point percentage", "Historical rolling feature only"],
    ["LEAGUE_POINTS_RATE_DIFF", "Premier League", "Difference in historical league points per game", "Previous matches only"],
    ["SHOTS_DIFF", "Premier League", "Difference in rolling shots where available", "Historical rolling feature only"],
    ["SHOTS_ON_TARGET_DIFF", "Premier League", "Difference in rolling shots on target where available", "Historical rolling feature only"],
], columns=[
    "feature",
    "sport",
    "description",
    "leakage_safe_calculation",
])

FEATURE_DICTIONARY_OUTPUT = (
    DOCS_DIR
    / "week6_feature_dictionary.csv"
)

feature_dictionary.to_csv(
    FEATURE_DICTIONARY_OUTPUT,
    index=False,
)

# ------------------------------------------------------------
# 11. Write Week 6 methodology/reproduction documentation
# ------------------------------------------------------------

week6_notes = f"""# Against All Odds — Week 6 Feature Engineering

Generated: {datetime.now(timezone.utc).isoformat(timespec="seconds")}

## Week 6 deliverable

Three leakage-safe model-ready datasets were created:

- {NBA_OUTPUT.relative_to(PROJECT_ROOT)}
- {NFL_OUTPUT.relative_to(PROJECT_ROOT)}
- {EPL_OUTPUT.relative_to(PROJECT_ROOT)}

## Core feature groups

### Team strength
- Pre-game win percentage
- Pre-game Elo
- Historical point/goal differential

### Recent form
- Last 3
- Last 5
- Last 10 where appropriate

### Offense
- Historical points/goals scored
- Recent scoring
- NBA efficiency metrics where available
- Premier League shots/shots-on-target where available

### Defense
- Historical points/goals allowed
- Recent defensive performance
- NBA defensive rating where available

### Context
- Home-team perspective
- Rest
- Current opponent Elo
- Recent schedule strength

### Matchup differences
- ELO_DIFF
- WIN_PCT_DIFF
- FORM_DIFF
- POINT_GOAL_DIFF
- OFFENSE_DIFF
- DEFENSE_DIFF
- REST_DIFF
- SCHEDULE_STRENGTH_DIFF

## Leakage prevention

Every historical expanding or rolling feature applies shift(1) before the
calculation. The current game's final score, result, and box/match statistics
cannot enter that game's pre-game predictor values.

The final model-ready CSV files intentionally keep HOME_WIN as the target while
excluding current-game final scores and current-game box/match statistics from
the predictor columns.

## Sport-specific handling

### NBA
HOME_WIN is binary. Historical NBA efficiency statistics are used only when
those fields are present in the clean Week 3 data.

### NFL
A tie contributes 0.5 to historical team-strength and Elo updates. Tied games
are excluded from the final binary HOME_WIN modeling dataset.

### Premier League
HOME_WIN equals 1 only for a home win. Draws and away wins equal 0 for the
binary target. Draws contribute 0.5 to Elo updates.

## Next step

Week 7 will use these model-ready datasets for exploratory data analysis and
baseline predictions.
"""

WEEK6_NOTES_OUTPUT = (
    DOCS_DIR
    / "week6_feature_engineering_notes.md"
)

WEEK6_NOTES_OUTPUT.write_text(
    week6_notes,
    encoding="utf-8",
)

# README text that can be copied into the main project README.
readme_addition = """## Week 6 — Feature Engineering

Created leakage-safe pre-game features for the NBA, NFL, and Premier League.
Features include pre-game win percentage, Elo, point/goal differential, recent
form, offense, defense, rest, opponent strength, schedule strength, and matchup
difference variables. All rolling historical calculations use `shift(1)` so
the current game cannot use information from itself. Three model-ready datasets
were produced for Week 7 exploratory analysis and baseline modeling.
"""

README_ADDITION_OUTPUT = (
    DOCS_DIR
    / "week6_README_addition.md"
)

README_ADDITION_OUTPUT.write_text(
    readme_addition,
    encoding="utf-8",
)

print("\n✅ Documentation saved")
print(" -", FEATURE_DICTIONARY_OUTPUT)
print(" -", WEEK6_NOTES_OUTPUT)
print(" -", README_ADDITION_OUTPUT)

# ------------------------------------------------------------
# 12. Create a Week 6 GitHub-upload ZIP
# ------------------------------------------------------------

GITHUB_RUNTIME_FILES = [
    NBA_OUTPUT,
    NFL_OUTPUT,
    EPL_OUTPUT,
    FEATURE_DICTIONARY_OUTPUT,
    WEEK6_NOTES_OUTPUT,
    README_ADDITION_OUTPUT,
    QUALITY_OUTPUT,
    SPLIT_OUTPUT,
    MISSINGNESS_OUTPUT,
]

WEEK6_ZIP = (
    PROJECT_ROOT
    / "Against_All_Odds_Week6_GitHub_Upload.zip"
)

if WEEK6_ZIP.exists():
    WEEK6_ZIP.unlink()

with zipfile.ZipFile(
    WEEK6_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in GITHUB_RUNTIME_FILES:
        if path.exists():
            archive.write(
                path,
                path.relative_to(PROJECT_ROOT),
            )

print("\n✅ Week 6 GitHub ZIP created:")
print(WEEK6_ZIP)

# ------------------------------------------------------------
# 13. Final Week 6 deliverable checks
# ------------------------------------------------------------

deliverables = {
    "NBA model-ready dataset": NBA_OUTPUT.exists(),
    "NFL model-ready dataset": NFL_OUTPUT.exists(),
    "Premier League model-ready dataset": EPL_OUTPUT.exists(),
    "Feature dictionary": FEATURE_DICTIONARY_OUTPUT.exists(),
    "Feature-engineering notes": WEEK6_NOTES_OUTPUT.exists(),
    "README addition": README_ADDITION_OUTPUT.exists(),
    "Quality report": QUALITY_OUTPUT.exists(),
    "Split summary": SPLIT_OUTPUT.exists(),
    "Missingness report": MISSINGNESS_OUTPUT.exists(),
    "GitHub upload ZIP": WEEK6_ZIP.exists(),
}

deliverable_status = pd.DataFrame([
    {
        "deliverable": name,
        "created": bool(created),
    }
    for name, created in deliverables.items()
])

print("\nFinal Week 6 deliverable status:")
display(deliverable_status)

if not deliverable_status["created"].all():
    raise RuntimeError(
        "One or more Week 6 deliverables were not created."
    )

print("\n" + "=" * 78)
print("AGAINST ALL ODDS — WEEK 6 COMPLETE")
print("=" * 78)
print("✅ Leakage-safe feature engineering complete")
print("✅ NBA model-ready dataset complete")
print("✅ NFL model-ready dataset complete")
print("✅ Premier League model-ready dataset complete")
print("✅ Week 6 documentation and quality reports complete")

# ------------------------------------------------------------
# 14. EXACT GITHUB PUSH LIST
# ------------------------------------------------------------

github_push_list = [
    "notebooks/07_feature_engineering.ipynb",
    "data/model_ready/nba/nba_model_ready_2021_22_to_2025_26.csv",
    "data/model_ready/nfl/nfl_model_ready_2021_to_2025.csv",
    "data/model_ready/soccer/premier_league_model_ready_2021_22_to_2025_26.csv",
    "docs/week6_feature_dictionary.csv",
    "docs/week6_feature_engineering_notes.md",
    "docs/week6_README_addition.md",
    "results/week6_feature_engineering_quality_report.csv",
    "results/week6_model_ready_split_summary.csv",
    "results/week6_model_ready_missingness.csv",
]

print("\nWHAT TO PUSH TO GITHUB FOR WEEK 6")
print("-" * 78)

for item in github_push_list:
    print(" -", item)

print(
    "\nAlso update the main README.md by copying the text from "
    "docs/week6_README_addition.md."
)

print(
    "\nDo NOT push the Week6 ZIP itself unless you specifically want "
    "an archive in the repository. Extract it and upload the files "
    "into the matching GitHub folders."
)

print(
    "\nNo requirements.txt change is needed for Week 6 because this "
    "code uses packages already required by the project."
)

print("\nNext: Week 7 — Exploratory Data Analysis and Baselines.")

# Optional convenience download in Colab.
if running_in_colab():
    print(
        "\nTo download the Week 6 upload ZIP immediately, run:\n"
        "from google.colab import files\n"
        "files.download(str(WEEK6_ZIP))"
    )


✅ Against All Odds — Week 6
Project root: /content/againstallodds

The following Week 3–5 clean CSV files are missing:
 - nba_games_clean_2021_22_to_2025_26.csv
 - nba_team_game_logs_clean_2021_22_to_2025_26.csv
 - nfl_games_clean_2021_to_2025.csv
 - nfl_team_game_logs_clean_2021_to_2025.csv
 - premier_league_games_clean_2021_22_to_2025_26.csv
 - premier_league_team_games_clean_2021_22_to_2025_26.csv

Upload the missing CSV files or a ZIP containing them. Colab will continue after the upload.


Saving ZIP WEEK 6.zip to ZIP WEEK 6.zip

✅ Required Week 6 inputs located:
 - nba_games: /content/week6_inputs/New folder/nba_games_clean_2021_22_to_2025_26.csv
 - nba_team_games: /content/week6_inputs/New folder/nba_team_game_logs_clean_2021_22_to_2025_26.csv
 - nfl_games: /content/week6_inputs/New folder/nfl_games_clean_2021_to_2025.csv
 - nfl_team_games: /content/week6_inputs/New folder/nfl_team_game_logs_clean_2021_to_2025.csv
 - epl_games: /content/week6_inputs/New folder/premier_league_games_clean_2021_22_to_2025_26.csv
 - epl_team_games: /content/week6_inputs/New folder/premier_league_team_games_clean_2021_22_to_2025_26.csv

✅ Input schema checks passed
NBA: (6140, 72) | team-game: (12280, 102)
NFL: (1359, 25) | team-game: (2718, 16)
EPL: (1900, 28) | team-game: (3800, 25)

✅ NBA shift(1) leakage audit passed
✅ NBA: 6,140 clean games -> 6,061 model-ready games

NBA model-ready preview:


,game_id,game_date,season_start,season_end,season_label,data_split,home_team_abbr,home_team_name,away_team_abbr,away_team_name,...,AWAY_OPPONENT_ELO_PRE,HOME_SCHEDULE_STRENGTH_LAST5,AWAY_SCHEDULE_STRENGTH_LAST5,SCHEDULE_STRENGTH_DIFF,HOME_ADVANTAGE,OFF_RATING_DIFF,DEF_RATING_DIFF,NET_RATING_DIFF,EFG_DIFF,THREE_PT_DIFF
0,0022100017,2021-10-22,2021,2022,2021-22,TRAIN,CLE,Cleveland Cavaliers,CHA,Charlotte Hornets,...,1490.0,1500.0,1500.0,0.0,1,0.3,-11.7,-11.4,0.090,-0.051
1,0022100018,2021-10-22,2021,2022,2021-22,TRAIN,ORL,Orlando Magic,NYK,New York Knicks,...,1490.0,1500.0,1500.0,0.0,1,-8.3,-20.0,-28.3,-0.108,-0.068
2,0022100019,2021-10-22,2021,2022,2021-22,TRAIN,WAS,Washington Wizards,IND,Indiana Pacers,...,1510.0,1500.0,1500.0,0.0,1,-21.5,38.4,16.9,-0.086,-0.177
3,0022100020,2021-10-22,2021,2022,2021-22,TRAIN,BOS,Boston Celtics,TOR,Toronto Raptors,...,1490.0,1500.0,1500.0,0.0,1,30.5,-17.9,12.6,0.155,0.162
4,0022100021,2021-10-22,2021,2022,2021-22,TRAIN,PHI,Philadelphia 76ers,BKN,Brooklyn Nets,...,1510.0,1500.0,1500.0,0.0,1,16.2,26.5,42.7,0.052,-0.011



✅ NFL shift(1) leakage audit passed
✅ NFL: 1,359 clean games -> 1,276 model-ready games

NFL note: ties update historical Elo and team strength using 0.5, but tied games are excluded from the final binary HOME_WIN model-ready dataset.

NFL model-ready preview:


,game_id,game_date,season,week,data_split,home_team,away_team,HOME_WIN,HOME_ELO_PRE,AWAY_ELO_PRE,...,DEFENSE_DIFF,HOME_REST_DAYS,AWAY_REST_DAYS,REST_DIFF,HOME_OPPONENT_ELO_PRE,AWAY_OPPONENT_ELO_PRE,HOME_SCHEDULE_STRENGTH_LAST5,AWAY_SCHEDULE_STRENGTH_LAST5,SCHEDULE_STRENGTH_DIFF,HOME_ADVANTAGE
0,2021_02_NYG_WAS,2021-09-16,2021,2,TRAIN,WAS,NYG,1,1490.0,1490.0,...,7.0,4.0,4.0,0.0,1490.0,1490.0,1500.0,1500.0,0.0,1
1,2021_02_ATL_TB,2021-09-19,2021,2,TRAIN,TB,ATL,1,1510.0,1490.0,...,3.0,10.0,7.0,3.0,1490.0,1510.0,1500.0,1500.0,0.0,1
2,2021_02_BUF_MIA,2021-09-19,2021,2,TRAIN,MIA,BUF,0,1510.0,1490.0,...,7.0,7.0,7.0,0.0,1490.0,1510.0,1500.0,1500.0,0.0,1
3,2021_02_CIN_CHI,2021-09-19,2021,2,TRAIN,CHI,CIN,1,1490.0,1510.0,...,-10.0,7.0,7.0,0.0,1510.0,1490.0,1500.0,1500.0,0.0,1
4,2021_02_DAL_LAC,2021-09-19,2021,2,TRAIN,LAC,DAL,0,1510.0,1490.0,...,15.0,7.0,10.0,-3.0,1490.0,1510.0,1500.0,1500.0,0.0,1


...


Model-ready split summary:


,sport,data_split,games,home_win_rate
0,NBA,TEST,1209,0.553350
1,NBA,TRAIN,3643,0.555586
2,NBA,VALIDATION,1209,0.547560
3,NFL,TEST,255,0.537255
4,NFL,TRAIN,765,0.551634
5,NFL,VALIDATION,256,0.523438
6,Premier League,TEST,370,0.424324
7,Premier League,TRAIN,1110,0.456757
8,Premier League,VALIDATION,370,0.408108



✅ Week 6 datasets and reports saved
 - /content/againstallodds/data/model_ready/nba/nba_model_ready_2021_22_to_2025_26.csv
 - /content/againstallodds/data/model_ready/nfl/nfl_model_ready_2021_to_2025.csv
 - /content/againstallodds/data/model_ready/soccer/premier_league_model_ready_2021_22_to_2025_26.csv
 - /content/againstallodds/results/week6_feature_engineering_quality_report.csv
 - /content/againstallodds/results/week6_model_ready_split_summary.csv
 - /content/againstallodds/results/week6_model_ready_missingness.csv

✅ Documentation saved
 - /content/againstallodds/docs/week6_feature_dictionary.csv
 - /content/againstallodds/docs/week6_feature_engineering_notes.md
 - /content/againstallodds/docs/week6_README_addition.md

✅ Week 6 GitHub ZIP created:
/content/againstallodds/Against_All_Odds_Week6_GitHub_Upload.zip

Final Week 6 deliverable status:


,deliverable,created
0,NBA model-ready dataset,True
1,NFL model-ready dataset,True
2,Premier League model-ready dataset,True
3,Feature dictionary,True
4,Feature-engineering notes,True
5,README addition,True
6,Quality report,True
7,Split summary,True
8,Missingness report,True
9,GitHub upload ZIP,True



AGAINST ALL ODDS — WEEK 6 COMPLETE
✅ Leakage-safe feature engineering complete
✅ NBA model-ready dataset complete
✅ NFL model-ready dataset complete
✅ Premier League model-ready dataset complete
✅ Week 6 documentation and quality reports complete

WHAT TO PUSH TO GITHUB FOR WEEK 6
------------------------------------------------------------------------------
 - notebooks/07_feature_engineering.ipynb
 - data/model_ready/nba/nba_model_ready_2021_22_to_2025_26.csv
 - data/model_ready/nfl/nfl_model_ready_2021_to_2025.csv
 - data/model_ready/soccer/premier_league_model_ready_2021_22_to_2025_26.csv
 - docs/week6_feature_dictionary.csv
 - docs/week6_feature_engineering_notes.md
 - docs/week6_README_addition.md
 - results/week6_feature_engineering_quality_report.csv
 - results/week6_model_ready_split_summary.csv
 - results/week6_model_ready_missingness.csv

Also update the main README.md by copying the text from docs/week6_README_addition.md.

Do NOT push the Week6 ZIP itself unless you speci

In [4]:

# ============================================================
# FIXED WEEK 6 DOWNLOAD CELL
# Self-contained: does NOT require WEEK6_ZIP to already exist.
# ============================================================

from pathlib import Path
import zipfile
import sys

# Files that Week 6 should have created before download.
REQUIRED_WEEK6_FILES = [
    "data/model_ready/nba/nba_model_ready_2021_22_to_2025_26.csv",
    "data/model_ready/nfl/nfl_model_ready_2021_to_2025.csv",
    "data/model_ready/soccer/premier_league_model_ready_2021_22_to_2025_26.csv",
    "docs/week6_feature_dictionary.csv",
    "docs/week6_feature_engineering_notes.md",
    "docs/week6_README_addition.md",
    "results/week6_feature_engineering_quality_report.csv",
    "results/week6_model_ready_split_summary.csv",
    "results/week6_model_ready_missingness.csv",
]

# Search the common project locations used by this capstone.
candidate_roots = [
    Path("/content/againstallodds"),
    Path("/content/Against-All-Odds"),
    Path("/content/drive/MyDrive/againstallodds"),
    Path("/content/drive/MyDrive/Against-All-Odds"),
    Path.cwd().resolve(),
]

# Keep only existing locations.
candidate_roots = [p for p in candidate_roots if p.exists()]

if not candidate_roots:
    raise FileNotFoundError(
        "Could not find the Against All Odds project folder in this Colab runtime."
    )

# Pick the location containing the most Week 6 output files.
def count_outputs(root):
    return sum((root / rel).exists() for rel in REQUIRED_WEEK6_FILES)

PROJECT_ROOT = max(candidate_roots, key=count_outputs)

print("Using project folder:", PROJECT_ROOT)

missing = [
    rel
    for rel in REQUIRED_WEEK6_FILES
    if not (PROJECT_ROOT / rel).exists()
]

if missing:
    print("\n❌ The ZIP cannot be created yet.")
    print("These Week 6 files are missing:")
    for rel in missing:
        print(" -", rel)

    raise FileNotFoundError(
        "\nRun the full Week 6 feature-engineering code first. "
        "When the final Week 6 status says COMPLETE, rerun this download cell."
    )

# Create a fresh ZIP in /content so Colab can download it easily.
ZIP_PATH = Path("/content/Against_All_Odds_Week6_GitHub_Upload.zip")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:

    for rel in REQUIRED_WEEK6_FILES:
        source = PROJECT_ROOT / rel
        archive.write(source, arcname=rel)

    # Include a plain-text GitHub upload checklist.
    push_list = """AGAINST ALL ODDS — WEEK 6 GITHUB PUSH LIST

Push these files:

notebooks/
  07_feature_engineering.ipynb

data/model_ready/nba/
  nba_model_ready_2021_22_to_2025_26.csv

data/model_ready/nfl/
  nfl_model_ready_2021_to_2025.csv

data/model_ready/soccer/
  premier_league_model_ready_2021_22_to_2025_26.csv

docs/
  week6_feature_dictionary.csv
  week6_feature_engineering_notes.md
  week6_README_addition.md

results/
  week6_feature_engineering_quality_report.csv
  week6_model_ready_split_summary.csv
  week6_model_ready_missingness.csv

Also copy the Week 6 README addition into the main README.md.
"""

    checklist_path = Path("/content/WEEK6_GITHUB_PUSH_LIST.txt")
    checklist_path.write_text(push_list, encoding="utf-8")
    archive.write(checklist_path, arcname="WEEK6_GITHUB_PUSH_LIST.txt")

print("\n✅ ZIP created successfully:")
print(ZIP_PATH)
print(f"ZIP size: {ZIP_PATH.stat().st_size / (1024 * 1024):.2f} MB")

# Download directly in Google Colab.
try:
    from google.colab import files
    files.download(str(ZIP_PATH))
    print("\n✅ Download started.")
except ImportError:
    print("\nNot running in Google Colab.")
    print("The ZIP is saved at:", ZIP_PATH)


Using project folder: /content/againstallodds

✅ ZIP created successfully:
/content/Against_All_Odds_Week6_GitHub_Upload.zip
ZIP size: 1.37 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download started.
